# 2차 — 최종 푸시: 풍부한 피처셋 CatBoost + ECDF 선형회귀 스태킹

## 목표: 실제 제출 0.742 돌파

## 0순위 (이 노트북 돌리기 전에)
**`1차_final_team_blend_submit_FIXED`(OOF 0.74071)를 아직 실제 제출 안 하셨으면 지금 먼저 제출하세요.**
OOF→실제 격차가 보통 +0.0012~0.0015였으니, 이미 0.742 근처일 수 있습니다.

## 이 노트북의 두 가지 새 시도
1. **CatBoost + 풍부한 피처셋** (Keep-NaN/클리핑/안전비율/실효나이/시술유형 토큰화/기증자 회춘 격차) —
   지금까지 LightGBM으로만 테스트했던 피처셋을 CatBoost(NaN 네이티브 처리가 더 강함)에 적용
2. **ECDF 랭크변환 + 선형회귀 메타러너** — 기존 블렌딩(가중치 0~1, 합=1 제약)과 달리
   **음수 가중치 허용** → 모델간 공유 노이즈 상쇄 가능. blend_cache의 모든 기존 후보 +
   이번에 새로 만드는 모델까지 전부 포함해서 스태킹.

## "무한 반복" 가이드
이 노트북 끝에 `run_new_model(name, build_fn)` 패턴을 만들어둘게요. 새 모델 아이디어가 생기면
그 패턴대로 함수만 추가하고 다시 스태킹 셀을 돌리면 됩니다 — 후보가 늘어날수록 ECDF
스태킹이 자동으로 더 많은 조합을 고려해요.

## 누수 방지
카테고리/비율 모두 train 기준만 사용, ECDF는 매 fold의 train 부분으로만 fit.

---
**저장 위치:** `experiment_history/2차/2차_final_push_stacking.ipynb`


In [1]:
import json
import time
import glob
from pathlib import Path

import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score
import warnings

warnings.filterwarnings("ignore")


In [2]:
DATA_DIR = Path("../../data")
SHARED_DIR = Path("..")
BLEND_DIR = SHARED_DIR / "blend_cache"
SUBMIT_DIR = Path("../submit file")
TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
TARGET_COL = "임신 성공 여부"
DEAD_COLS = ["불임 원인 - 여성 요인", "난자 채취 경과일"]
N_SPLITS = 5

EMBRYO_COUNT_COLS = ["총 생성 배아 수","미세주입된 난자 수","미세주입에서 생성된 배아 수","이식된 배아 수",
    "미세주입 배아 이식 수","저장된 배아 수","미세주입 후 저장된 배아 수","해동된 배아 수",
    "해동 난자 수","수집된 신선 난자 수","저장된 신선 난자 수","혼합된 난자 수",
    "파트너 정자와 혼합된 난자 수","기증자 정자와 혼합된 난자 수"]
EMBRYO_BINARY_COLS = ["단일 배아 이식 여부","착상 전 유전 진단 사용 여부","동결 배아 사용 여부",
    "신선 배아 사용 여부","기증 배아 사용 여부","대리모 여부"]
CLIP_RULES = {"총 생성 배아 수":40,"수집된 신선 난자 수":40,"미세주입된 난자 수":45,
    "혼합된 난자 수":40,"저장된 배아 수":30,"배아 이식 경과일":7,"난자 혼합 경과일":7,
    "배아 해동 경과일":2,"난자 해동 경과일":1}
RATIO_SPECS = [
    ("총 생성 배아 수", "혼합된 난자 수", "fertilization_rate"),
    ("미세주입에서 생성된 배아 수", "미세주입된 난자 수", "icsi_fertilization_rate"),
    ("이식된 배아 수", "총 생성 배아 수", "embryo_utilization_rate"),
    ("저장된 배아 수", "총 생성 배아 수", "embryo_freezing_rate"),
    ("혼합된 난자 수", "수집된 신선 난자 수", "oocyte_utilization_rate"),
    ("이식된 배아 수", "해동된 배아 수", "thawed_embryo_transfer_ratio"),
]


## 1. 풍부한 피처셋 빌드 (Keep-NaN + 클리핑 + 비율 + 실효나이 + 시술유형 토큰화)

In [3]:
train_raw = pd.read_csv(TRAIN_PATH)
test_raw_full = pd.read_csv(TEST_PATH)
test_ids = test_raw_full["ID"]
y = train_raw[TARGET_COL].values.astype(int)


def safe_ratio(df, numerator, denominator, new_col):
    df = df.copy()
    can = df[numerator].notna() & df[denominator].notna() & (df[denominator] > 0)
    df[f"{new_col}_available"] = can.astype(int)
    df[new_col] = np.where(can, df[numerator] / df[denominator], np.nan)
    return df


def build_full_features(df):
    df = df.copy()
    df = df.drop(columns=[c for c in DEAD_COLS if c in df.columns])
    df["is_DI"] = (df["시술 유형"] == "DI").astype(int)
    df["froze_embryo"] = df["동결 배아 사용 여부"].fillna(0).astype(int)

    embryo_cols_present = [c for c in (EMBRYO_COUNT_COLS + EMBRYO_BINARY_COLS) if c in df.columns]
    for col in embryo_cols_present:
        df[f"{col}_missing"] = df[col].isna().astype(int)
        is_di_missing = (df["시술 유형"] == "DI") & df[col].isna()
        df[f"{col}_not_applicable_DI"] = is_di_missing.astype(int)

    for col, upper in CLIP_RULES.items():
        if col in df.columns:
            df[f"{col}_high_flag"] = (df[col] > upper).astype(int)
            df[col] = df[col].clip(upper=upper)

    if "배아 이식 경과일" in df.columns:
        df["transfer_day_0_1_flag"] = df["배아 이식 경과일"].isin([0, 1]).astype(int)
        df["transfer_day_3_flag"] = (df["배아 이식 경과일"] == 3).astype(int)
        df["transfer_day_5_or_more_flag"] = (df["배아 이식 경과일"] >= 5).astype(int)

    for num, den, new in RATIO_SPECS:
        if num in df.columns and den in df.columns:
            df = safe_ratio(df, num, den, new)

    # 실효 나이 (기증 난자면 기증자 나이로 교체)
    patient_mid = {"만18-34세":31,"만35-37세":36,"만38-39세":38.5,"만40-42세":41,
                    "만43-44세":43.5,"만45-50세":47.5,"알 수 없음":np.nan}
    donor_mid = {"만20세 이하":20,"만21-25세":23,"만26-30세":28,"만31-35세":33,
                 "만36-40세":38,"만41-45세":43,"알 수 없음":np.nan}
    if "난자 출처" in df.columns and "시술 당시 나이" in df.columns:
        df["patient_age_mid"] = df["시술 당시 나이"].map(patient_mid)
        donor_age_mid = df["난자 기증자 나이"].map(donor_mid) if "난자 기증자 나이" in df.columns else pd.Series(np.nan, index=df.index)
        donor_known = (df["난자 출처"] == "기증 제공") & donor_age_mid.notna()
        df["effective_maternal_age_mid"] = df["patient_age_mid"]
        df.loc[donor_known, "effective_maternal_age_mid"] = donor_age_mid[donor_known]
        df["donor_rejuvenation_gap"] = 0.0
        df.loc[donor_known, "donor_rejuvenation_gap"] = (
            df.loc[donor_known, "patient_age_mid"] - donor_age_mid[donor_known]
        )

    # 시술유형 세부 토큰화
    if "특정 시술 유형" in df.columns:
        s = df["특정 시술 유형"].astype(str)
        for token in ["IVF", "ICSI", "IUI", "ICI", "GIFT", "FER", "Generic DI", "IVI", "BLASTOCYST", "AH"]:
            safe = token.lower().replace(" ", "_")
            df[f"spec_has_{safe}"] = s.str.contains(token, regex=False, na=False).astype(int)

    # 인터랙션 (카테고리 결합)
    if {"시술 당시 나이", "난자 출처"}.issubset(df.columns):
        df["age_oocyte_source"] = df["시술 당시 나이"].astype(str) + "_" + df["난자 출처"].astype(str)

    return df


train_feat = build_full_features(train_raw.drop(columns=["ID"]))
test_feat = build_full_features(test_raw_full.drop(columns=["ID"]))

X_train_full = train_feat.drop(columns=[TARGET_COL])
X_test_full = test_feat.copy()

# ★ 대회 규칙 준수: 카테고리는 train 기준만. 범주형 NaN(원래 결측 + test에만 있는 새 값 둘 다)은
# CatBoost가 못 받으므로 전부 sentinel 문자열로 흡수
# 범주형 처리 (train 기준만) — sentinel로 결측 흡수

SENTINEL = "정보없음"
obj_cols = X_train_full.select_dtypes(include=["object"]).columns.tolist()
for col in obj_cols:
    X_train_full[col] = X_train_full[col].fillna(SENTINEL)
    X_test_full[col] = X_test_full[col].fillna(SENTINEL)
    train_categories = sorted(set(X_train_full[col].astype(str).unique()) | {SENTINEL})
    X_train_full[col] = pd.Categorical(X_train_full[col].astype(str), categories=train_categories)
    X_test_full[col] = pd.Categorical(X_test_full[col].astype(str), categories=train_categories)
    X_test_full[col] = X_test_full[col].fillna(SENTINEL)
X_test_full = X_test_full[X_train_full.columns]

print(f"풍부한 피처셋: train {X_train_full.shape}, test {X_test_full.shape}, 범주형 {len(obj_cols)}개")


풍부한 피처셋: train (256351, 145), test (90067, 145), 범주형 21개


## 4. 기존 blend_cache 전체 자동 스캔 (새 후보 포함)

In [4]:
MANUAL_TEST_MAP = {"oof_10seed_bagged.npy": "test_lgbm_bagged.npy"}
V5_DIR = SHARED_DIR / "팀원파일" / "김영혜"
V5_OOF_PATH = V5_DIR / "v5_ensemble_oof.npy"
V5_SUBMISSION_PATH = V5_DIR / "submission_v5_imputed.csv"

all_npy = sorted(glob.glob(str(BLEND_DIR / "*.npy")))
oof_candidates = [f for f in all_npy if "oof" in Path(f).stem.lower()]

models = {}
for f in oof_candidates:
    fname = Path(f).name
    stem = Path(f).stem
    oof_arr = np.load(f)
    if len(oof_arr) != len(y):
        continue
    if stem.startswith("oof_"):
        name = stem[len("oof_"):]
        default_test = BLEND_DIR / f"test_{name}.npy"
    elif stem.endswith("_oof"):
        name = stem[: -len("_oof")]
        default_test = BLEND_DIR / f"{name}_test.npy"
    else:
        name = stem
        default_test = None
    test_f = BLEND_DIR / MANUAL_TEST_MAP[fname] if fname in MANUAL_TEST_MAP else default_test
    entry = {"oof": oof_arr}
    if test_f is not None and test_f.exists():
        entry["test"] = np.load(test_f)
    models[name] = entry

if V5_OOF_PATH.exists():
    v5_oof = np.load(V5_OOF_PATH)
    if len(v5_oof) == len(y):
        entry = {"oof": v5_oof}
        if V5_SUBMISSION_PATH.exists():
            entry["test"] = pd.read_csv(V5_SUBMISSION_PATH)["probability"].values
        models["v5_ensemble"] = entry

# 안전장치
for n in list(models.keys()):
    if n == "y" or roc_auc_score(y, models[n]["oof"]) >= 0.995:
        del models[n]

model_names_with_test = [n for n in models if "test" in models[n]]
print(f"test 예측 있는 전체 후보 수: {len(model_names_with_test)}")
print(f"{'후보':<28} | {'OOF AUC':>8}")
print("-" * 42)
for n in sorted(model_names_with_test, key=lambda x: -roc_auc_score(y, models[x]["oof"])):
    print(f"{n:<28} | {roc_auc_score(y, models[n]['oof']):>8.5f}")


test 예측 있는 전체 후보 수: 13
후보                           |  OOF AUC
------------------------------------------
pseudolabel_PENDING_APPROVAL |  0.74088
donor_cat                    |  0.74076
v5_ensemble                  |  0.74043
10seed_bagged                |  0.74036
alpha_cat                    |  0.74033
xgboost_bagged               |  0.74010
catboost_bagged              |  0.74005
feature_subspace             |  0.73973
xgb_rankpairwise             |  0.73939
lgbm_lambdarank              |  0.73748
lgbm_keepnan                 |  0.73648
mlp                          |  0.73313
lgbm_spw2_richfeat           |  0.73297


## 5. ECDF 랭크변환 + 선형회귀 메타러너 (음수 가중치 허용, fold-safe)

In [5]:
class ECDFReference:
    """fold-train 분포로 fit. transform은 그 분포 기준 평균순위/n (동점 보정)."""
    def __init__(self, ref):
        self.sorted = np.sort(np.asarray(ref, dtype=float))
        self.n = max(len(self.sorted), 1)

    def transform(self, x):
        x = np.asarray(x, dtype=float)
        left = np.searchsorted(self.sorted, x, side="left")
        right = np.searchsorted(self.sorted, x, side="right")
        return (left + right) / 2.0 / self.n


def make_ecdf_linear_stack(names, models, y, n_splits=N_SPLITS, seed=42):
    oof_matrix = np.column_stack([models[n]["oof"] for n in names])
    has_test = all("test" in models[n] for n in names)
    test_matrix = np.column_stack([models[n]["test"] for n in names]) if has_test else None

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    oof_pred = np.zeros(len(y))
    test_pred = np.zeros(test_matrix.shape[0]) if has_test else None
    ncol = len(names)

    for tr, va in skf.split(oof_matrix, y):
        Xtr = np.empty((len(tr), ncol))
        Xva = np.empty((len(va), ncol))
        Xte = np.empty((test_matrix.shape[0], ncol)) if has_test else None
        for c in range(ncol):
            ref = ECDFReference(oof_matrix[tr, c])
            Xtr[:, c] = ref.transform(oof_matrix[tr, c])
            Xva[:, c] = ref.transform(oof_matrix[va, c])
            if has_test:
                Xte[:, c] = ref.transform(test_matrix[:, c])
        model = make_pipeline(StandardScaler(), LinearRegression())
        model.fit(Xtr, y[tr])
        oof_pred[va] = np.clip(model.predict(Xva), 0, 1)
        if has_test:
            test_pred += np.clip(model.predict(Xte), 0, 1) / n_splits

    auc = roc_auc_score(y, oof_pred)
    return oof_pred, test_pred, auc


# 여러 조합 시도 (1등 노트북처럼 subset 여러 개 비교)
all_names = model_names_with_test
print(f"전체 {len(all_names)}개 후보로 스태킹 시도\n")

stack_full_oof, stack_full_test, stack_full_auc = make_ecdf_linear_stack(all_names, models, y)
print(f"[전체 {len(all_names)}개] ECDF-LinearRegression 스택 OOF AUC: {stack_full_auc:.5f}")

# 상위 N개만 사용한 subset도 비교 (전체가 과적합 위험 있을 수 있음)
top_sorted = sorted(all_names, key=lambda x: -roc_auc_score(y, models[x]["oof"]))
for k in [3, 5, 7]:
    if k <= len(top_sorted):
        subset = top_sorted[:k]
        _, _, auc_k = make_ecdf_linear_stack(subset, models, y)
        print(f"[상위 {k}개: {subset}] OOF AUC: {auc_k:.5f}")

print()
print("비교 기준:")
print("  팀 블렌딩(2개 모델, 가중평균): 0.74071")
print("  1등 전체 아키텍처(7모델+5스택+3단계): 0.740973")


전체 13개 후보로 스태킹 시도

[전체 13개] ECDF-LinearRegression 스택 OOF AUC: 0.74108
[상위 3개: ['pseudolabel_PENDING_APPROVAL', 'donor_cat', 'v5_ensemble']] OOF AUC: 0.74102
[상위 5개: ['pseudolabel_PENDING_APPROVAL', 'donor_cat', 'v5_ensemble', '10seed_bagged', 'alpha_cat']] OOF AUC: 0.74105
[상위 7개: ['pseudolabel_PENDING_APPROVAL', 'donor_cat', 'v5_ensemble', '10seed_bagged', 'alpha_cat', 'xgboost_bagged', 'catboost_bagged']] OOF AUC: 0.74109

비교 기준:
  팀 블렌딩(2개 모델, 가중평균): 0.74071
  1등 전체 아키텍처(7모델+5스택+3단계): 0.740973


In [6]:
# =============================================================================
# Ridge 메타러너로 교체 + alpha 그리드 비교
# 놓을 위치: 2차_final_push_stacking.ipynb, "5. ECDF 랭크변환 + 선형회귀 메타러너" 셀 다음
# (ECDFReference, all_names, models, y, N_SPLITS, StandardScaler, make_pipeline,
#  roc_auc_score, np, StratifiedKFold는 이미 노트북에 정의되어 있음)
# =============================================================================
from sklearn.linear_model import Ridge, LinearRegression


def make_ecdf_ridge_stack(names, models, y, alpha=1.0, n_splits=N_SPLITS, seed=42):
    """LinearRegression 대신 Ridge(alpha)를 메타러너로 사용. alpha=0이면 LinearRegression과 동일."""
    oof_matrix = np.column_stack([models[n]["oof"] for n in names])
    has_test = all("test" in models[n] for n in names)
    test_matrix = np.column_stack([models[n]["test"] for n in names]) if has_test else None

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    oof_pred = np.zeros(len(y))
    test_pred = np.zeros(test_matrix.shape[0]) if has_test else None
    ncol = len(names)

    for tr, va in skf.split(oof_matrix, y):
        Xtr = np.empty((len(tr), ncol))
        Xva = np.empty((len(va), ncol))
        Xte = np.empty((test_matrix.shape[0], ncol)) if has_test else None
        for c in range(ncol):
            ref = ECDFReference(oof_matrix[tr, c])
            Xtr[:, c] = ref.transform(oof_matrix[tr, c])
            Xva[:, c] = ref.transform(oof_matrix[va, c])
            if has_test:
                Xte[:, c] = ref.transform(test_matrix[:, c])

        model = make_pipeline(StandardScaler(), Ridge(alpha=alpha) if alpha > 0 else LinearRegression())
        model.fit(Xtr, y[tr])
        oof_pred[va] = np.clip(model.predict(Xva), 0, 1)
        if has_test:
            test_pred += np.clip(model.predict(Xte), 0, 1) / n_splits

    auc = roc_auc_score(y, oof_pred)
    return oof_pred, test_pred, auc


# --- alpha 그리드 비교 (전체 12개 후보 기준) ---
print(f"{'alpha':>8} | {'OOF AUC':>8}")
print("-" * 21)
ridge_results = {}
for alpha in [0.0, 0.1, 0.3, 0.5, 1.0, 3.0, 10.0, 30.0]:
    _, _, auc = make_ecdf_ridge_stack(all_names, models, y, alpha=alpha)
    ridge_results[alpha] = auc
    label = "0.0 (=LinearRegression)" if alpha == 0.0 else str(alpha)
    print(f"{label:>8} | {auc:.5f}")

best_alpha = max(ridge_results, key=ridge_results.get)
print(f"\n최적 alpha={best_alpha}, OOF AUC={ridge_results[best_alpha]:.5f}")
print(f"참고: LinearRegression(alpha=0) 기준 = {ridge_results[0.0]:.5f}")

# --- 최적 alpha로 최종 test 예측 생성 (기존 6번 셀 대체) ---
FINAL_NAMES_RIDGE = all_names  # 필요하면 top_sorted[:N] 등으로 교체
final_oof_r, final_test_r, final_auc_r = make_ecdf_ridge_stack(
    FINAL_NAMES_RIDGE, models, y, alpha=best_alpha
)
print(f"\n최종(Ridge alpha={best_alpha}) OOF AUC: {final_auc_r:.5f}")

BLEND_DIR.mkdir(exist_ok=True)
np.save(BLEND_DIR / "ecdf_ridge_stack_test.npy", final_test_r)

SUBMIT_DIR.mkdir(exist_ok=True)
submission_r = pd.DataFrame({"ID": test_ids, "probability": final_test_r})
out_path_r = SUBMIT_DIR / f"2차_ecdf_ridge_stack_local{final_auc_r:.5f}.csv"
submission_r.to_csv(out_path_r, index=False)
print(f"저장 완료: {out_path_r}")

   alpha |  OOF AUC
---------------------
0.0 (=LinearRegression) | 0.74108
     0.1 | 0.74108
     0.3 | 0.74108
     0.5 | 0.74108
     1.0 | 0.74108
     3.0 | 0.74108
    10.0 | 0.74108
    30.0 | 0.74108

최적 alpha=0.3, OOF AUC=0.74108
참고: LinearRegression(alpha=0) 기준 = 0.74108

최종(Ridge alpha=0.3) OOF AUC: 0.74108
저장 완료: ../submit file/2차_ecdf_ridge_stack_local0.74108.csv


In [7]:
# === 5-1. Greedy + ECDF선형회귀 선택 (5번 셀 다음에 새 셀로 추가) ===
def greedy_ecdf_selection(pool_names, models, y, improve_threshold=0.00015):
    individual_aucs = {n: roc_auc_score(y, models[n]["oof"]) for n in pool_names}
    selected = [max(individual_aucs, key=individual_aucs.get)]
    remaining = [n for n in pool_names if n != selected[0]]
    current_auc = individual_aucs[selected[0]]
    print(f"시작: {selected[0]} (단독 AUC={current_auc:.5f})\n")

    history = [(list(selected), current_auc)]
    while remaining:
        best_candidate, best_new_auc = None, current_auc
        for cand in remaining:
            _, _, trial_auc = make_ecdf_linear_stack(selected + [cand], models, y)
            if trial_auc > best_new_auc:
                best_new_auc, best_candidate = trial_auc, cand
        improvement = best_new_auc - current_auc
        if best_candidate is None or improvement < improve_threshold:
            print(f"중단 (최선 개선폭 {improvement:+.5f} < {improve_threshold})")
            break
        selected.append(best_candidate)
        remaining.remove(best_candidate)
        current_auc = best_new_auc
        history.append((list(selected), current_auc))
        print(f"+ {best_candidate:<28} -> {current_auc:.5f}  (개선 {improvement:+.5f})")

    return selected, current_auc, history


greedy_selected, greedy_auc, greedy_history = greedy_ecdf_selection(all_names, models, y)
print(f"\n최종 선택: {greedy_selected}")
print(f"Greedy+ECDF 최종 OOF AUC: {greedy_auc:.5f}")
print(f"(참고: 전체 12개 한꺼번에 선형회귀 = 0.74095)")


시작: pseudolabel_PENDING_APPROVAL (단독 AUC=0.74088)

중단 (최선 개선폭 +0.00014 < 0.00015)

최종 선택: ['pseudolabel_PENDING_APPROVAL']
Greedy+ECDF 최종 OOF AUC: 0.74088
(참고: 전체 12개 한꺼번에 선형회귀 = 0.74095)


In [13]:
# =============================================================================
# pseudo-label을 13번째 후보로 포함한 ECDF 재스태킹
# ⚠️ 운영진 승인 전까지 이 셀의 결과 CSV는 제출하지 마세요
#
# 놓을 위치: 2차_final_push_stacking.ipynb 안에서 실행
# 순서: 1번 셀(피처빌드) -> 4번 셀(자동 스캔, 13개로 잡힐 것) -> 5번 셀(ECDFReference 등 정의)
#       -> 이 코드 (새 셀로 추가)
# 2,3번 셀(CatBoost/LightGBM 학습)은 또 안 돌려도 됨 — blend_cache에 이미 다 있음
# =============================================================================
from sklearn.linear_model import Ridge, LinearRegression

PENDING_DIR = Path("../pending_compliance_review")

# --- 13개 후보 확인 (pseudolabel이 들어왔는지 체크) ---
print(f"전체 후보 수: {len(all_names)}")
print(f"후보 목록: {all_names}")
if "pseudolabel_PENDING_APPROVAL" not in all_names:
    raise RuntimeError(
        "pseudolabel_PENDING_APPROVAL이 후보 목록에 없습니다 — "
        "4번 셀(자동 스캔)을 다시 실행했는지, blend_cache에 파일이 있는지 확인하세요."
    )
print("\n✅ pseudolabel_PENDING_APPROVAL이 후보 목록에 포함됨 — 13개 후보로 재스태킹 진행\n")


# --- ECDF + LinearRegression (전체 13개) ---
oof_13, test_13, auc_13 = make_ecdf_linear_stack(all_names, models, y)
print(f"[13개, LinearRegression] OOF AUC = {auc_13:.5f}")
print(f"(참고: pseudolabel 제외 12개 기준 = 0.74098)")
print(f"개선폭: {auc_13 - 0.74098:+.5f}\n")

# --- Ridge로도 재검증 (안정성 확인) ---
def make_ecdf_ridge_stack(names, models, y, alpha=1.0, n_splits=N_SPLITS, seed=42):
    oof_matrix = np.column_stack([models[n]["oof"] for n in names])
    test_matrix = np.column_stack([models[n]["test"] for n in names])
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    oof_pred = np.zeros(len(y))
    test_pred = np.zeros(test_matrix.shape[0])
    ncol = len(names)
    for tr, va in skf.split(oof_matrix, y):
        Xtr = np.empty((len(tr), ncol)); Xva = np.empty((len(va), ncol)); Xte = np.empty((test_matrix.shape[0], ncol))
        for c in range(ncol):
            ref = ECDFReference(oof_matrix[tr, c])
            Xtr[:, c] = ref.transform(oof_matrix[tr, c])
            Xva[:, c] = ref.transform(oof_matrix[va, c])
            Xte[:, c] = ref.transform(test_matrix[:, c])
        model = make_pipeline(StandardScaler(), Ridge(alpha=alpha) if alpha > 0 else LinearRegression())
        model.fit(Xtr, y[tr])
        oof_pred[va] = np.clip(model.predict(Xva), 0, 1)
        test_pred += np.clip(model.predict(Xte), 0, 1) / n_splits
    return oof_pred, test_pred, roc_auc_score(y, oof_pred)

print(f"{'alpha':>8} | {'OOF AUC':>8}")
print("-" * 21)
ridge_results_13 = {}
for alpha in [0.0, 0.5, 1.0, 3.0, 10.0]:
    _, _, auc = make_ecdf_ridge_stack(all_names, models, y, alpha=alpha)
    ridge_results_13[alpha] = auc
    print(f"{alpha:>8} | {auc:.5f}")

# --- 가중치 확인: pseudolabel이 실제로 얼마나 채택됐는지 ---
oof_matrix_13 = np.column_stack([models[n]["oof"] for n in all_names])
final_model = make_pipeline(StandardScaler(), LinearRegression())
ecdf_full = np.column_stack([
    ECDFReference(models[n]["oof"]).transform(models[n]["oof"]) for n in all_names
])
final_model.fit(ecdf_full, y)
coefs = final_model.named_steps["linearregression"].coef_
print("\n각 후보의 최종 계수 (절댓값 클수록 영향력 큼):")
for n, c in sorted(zip(all_names, coefs), key=lambda x: -abs(x[1])):
    marker = " <- pseudo-label" if n == "pseudolabel_PENDING_APPROVAL" else ""
    print(f"  {n:<32} {c:+.4f}{marker}")

# --- 결과 저장: 일반 제출 폴더가 아닌 승인 대기 폴더로 ---
PENDING_DIR.mkdir(exist_ok=True)
np.save(BLEND_DIR / "ecdf_stack_13_test_PENDING_APPROVAL.npy", test_13)

submission_13 = pd.DataFrame({"ID": test_ids, "probability": test_13})
out_path_13 = PENDING_DIR / f"2차_ecdf_stack_13_local{auc_13:.5f}_PENDING_APPROVAL.csv"
submission_13.to_csv(out_path_13, index=False)

print(f"\n저장 완료: {out_path_13}")
print("="*60)
print("⚠️  pseudo-label이 포함된 결과입니다. 운영진 승인 전까지 제출하지 마세요")
print("    (기존 안전한 최종 후보: 2차_ecdf_stack_local0.74098.csv는 그대로 유지)")
print("="*60)

전체 후보 수: 13
후보 목록: ['lgbm_lambdarank', '10seed_bagged', 'alpha_cat', 'catboost_bagged', 'donor_cat', 'feature_subspace', 'lgbm_keepnan', 'lgbm_spw2_richfeat', 'mlp', 'pseudolabel_PENDING_APPROVAL', 'xgboost_bagged', 'xgb_rankpairwise', 'v5_ensemble']

✅ pseudolabel_PENDING_APPROVAL이 후보 목록에 포함됨 — 13개 후보로 재스태킹 진행

[13개, LinearRegression] OOF AUC = 0.74108
(참고: pseudolabel 제외 12개 기준 = 0.74098)
개선폭: +0.00010

   alpha |  OOF AUC
---------------------
     0.0 | 0.74108
     0.5 | 0.74108
     1.0 | 0.74108
     3.0 | 0.74108
    10.0 | 0.74108

각 후보의 최종 계수 (절댓값 클수록 영향력 큼):
  pseudolabel_PENDING_APPROVAL     +0.0883 <- pseudo-label
  v5_ensemble                      +0.0465
  donor_cat                        +0.0428
  alpha_cat                        -0.0344
  catboost_bagged                  -0.0268
  10seed_bagged                    +0.0268
  xgb_rankpairwise                 -0.0252
  xgboost_bagged                   +0.0238
  lgbm_spw2_richfeat               +0.0149
  mlp                

In [8]:
PENDING_DIR = Path("../pending_compliance_review")  # 이 줄 추가

# Greedy 임계값 낮춰서 재시도 (pseudo-label이 들어간 13개 풀 기준)
greedy_selected_v2, greedy_auc_v2, greedy_history_v2 = greedy_ecdf_selection(
    all_names, models, y, improve_threshold=0.00003  # 기존 0.00015 -> 더 낮춤
)
print(f"\n최종 선택: {greedy_selected_v2}")
print(f"Greedy+ECDF(낮은 임계값) 최종 OOF AUC: {greedy_auc_v2:.5f}")
print(f"(참고: pseudo 단독 0.74088 / 13개 전체 0.74108 / 기존 12개 0.74098)")

# 이번 결과로 최종 제출 파일도 같이 생성 (여전히 PENDING 위치에)
oof_v2, test_v2, auc_v2 = make_ecdf_linear_stack(greedy_selected_v2, models, y)
PENDING_DIR.mkdir(exist_ok=True)
submission_v2 = pd.DataFrame({"ID": test_ids, "probability": test_v2})
out_path_v2 = PENDING_DIR / f"2차_greedy_lowthresh_local{auc_v2:.5f}_PENDING_APPROVAL.csv"
submission_v2.to_csv(out_path_v2, index=False)
print(f"\n저장 완료: {out_path_v2}")

시작: pseudolabel_PENDING_APPROVAL (단독 AUC=0.74088)

+ v5_ensemble                  -> 0.74102  (개선 +0.00014)
+ lgbm_lambdarank              -> 0.74108  (개선 +0.00006)
+ 10seed_bagged                -> 0.74112  (개선 +0.00004)
중단 (최선 개선폭 +0.00001 < 3e-05)

최종 선택: ['pseudolabel_PENDING_APPROVAL', 'v5_ensemble', 'lgbm_lambdarank', '10seed_bagged']
Greedy+ECDF(낮은 임계값) 최종 OOF AUC: 0.74112
(참고: pseudo 단독 0.74088 / 13개 전체 0.74108 / 기존 12개 0.74098)

저장 완료: ../pending_compliance_review/2차_greedy_lowthresh_local0.74112_PENDING_APPROVAL.csv


In [9]:
# pseudo-label 제외한 12개 풀로 낮은 임계값 재시도 (컴플라이언스 걱정 없는 안전한 시도)
names_12_only = [n for n in all_names if n != "pseudolabel_PENDING_APPROVAL"]

greedy_selected_safe, greedy_auc_safe, greedy_history_safe = greedy_ecdf_selection(
    names_12_only, models, y, improve_threshold=0.00003
)
print(f"\n최종 선택(pseudo 제외): {greedy_selected_safe}")
print(f"Greedy+ECDF(낮은 임계값, 12개 풀) 최종 OOF AUC: {greedy_auc_safe:.5f}")
print(f"(참고: 기존 12개 전체 0.74098 / pseudo 포함 4개 0.74112)")

oof_safe, test_safe, auc_safe = make_ecdf_linear_stack(greedy_selected_safe, models, y)
SUBMIT_DIR.mkdir(exist_ok=True)
submission_safe = pd.DataFrame({"ID": test_ids, "probability": test_safe})
out_path_safe = SUBMIT_DIR / f"2차_greedy_safe_local{auc_safe:.5f}.csv"
submission_safe.to_csv(out_path_safe, index=False)
print(f"\n저장 완료(제출 폴더, 컴플라이언스 문제 없음): {out_path_safe}")

시작: donor_cat (단독 AUC=0.74076)

+ v5_ensemble                  -> 0.74090  (개선 +0.00014)
+ lgbm_lambdarank              -> 0.74096  (개선 +0.00005)
+ 10seed_bagged                -> 0.74102  (개선 +0.00007)
중단 (최선 개선폭 +0.00002 < 3e-05)

최종 선택(pseudo 제외): ['donor_cat', 'v5_ensemble', 'lgbm_lambdarank', '10seed_bagged']
Greedy+ECDF(낮은 임계값, 12개 풀) 최종 OOF AUC: 0.74102
(참고: 기존 12개 전체 0.74098 / pseudo 포함 4개 0.74112)

저장 완료(제출 폴더, 컴플라이언스 문제 없음): ../submit file/2차_greedy_safe_local0.74102.csv


## 7. "무한 반복" — 새 모델 추가하는 법

새 모델 아이디어가 생기면:
1. OOF/test 배열 생성 (위 패턴처럼 5-fold CV)
2. `np.save(BLEND_DIR / "oof_{이름}.npy", oof)`, `np.save(BLEND_DIR / "test_{이름}.npy", test)`로 저장
3. 4번 셀(자동 스캔)부터 다시 실행 — 새 모델이 자동으로 풀에 잡힘
4. 5번 셀(ECDF 스태킹)을 다시 돌려서 조합에 도움 되는지 확인

후보가 많아질수록 5번 셀의 "상위 N개" 비교도 늘려보세요(top_sorted[:10] 등).
